In [1]:
import torch

In [5]:
torch.zeros(4, 8, 16).dim()

3

In [45]:
import math

import torch
from einops import einsum


class Linear(torch.nn.Module):
    weights: torch.Tensor

    def __init__(self, in_features: int, out_features: int, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        self.weights = torch.nn.parameter.Parameter(torch.empty(out_features, in_features, device=device, dtype=dtype))
        std = math.sqrt(2 / (in_features + out_features))
        torch.nn.init.trunc_normal_(tensor=self.weights, mean=0, std=std, a=-3 * std, b=3 * std)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return einsum(x, self.weights, "... in, out in -> ... out")

In [46]:
linear = Linear(1024, 1024)
linear.forward(torch.randn(1024, 1024))

list(linear.state_dict().keys())

['weights']

In [64]:
import torch
import math
from einops import einsum

class Embedding(torch.nn.Module):

    embeddings: torch.Tensor # (num_embeddings, embedding_dim)

    def __init__(self, num_embeddings: int, embedding_dim: int, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        self.embeddings = torch.nn.parameter.Parameter(torch.empty(num_embeddings, embedding_dim, device=device, dtype=dtype))
        torch.nn.init.trunc_normal_(tensor=self.embeddings, mean=0, std=1, a=-3, b=3)

    # token_ids: torch.LongTensor (batch_size, sequence_length)
    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        return self.embeddings[token_ids]

In [68]:
## embedding = Embedding(256, 1024)
embedding.forward(torch.randint(0, 256, (2, 5)))

tensor([[[ 1.3432, -1.9471,  0.2494,  ..., -1.3091,  0.2364, -1.5514],
         [-1.3648, -1.1296,  0.5849,  ..., -0.2595,  0.8650, -0.8553],
         [-0.2874,  0.1055, -0.5564,  ...,  0.1948,  0.7313,  1.2553],
         [ 1.7085,  0.6647,  0.0750,  ..., -0.1373,  0.5015, -0.1839],
         [-0.9239,  0.2824, -1.4718,  ...,  0.0264,  0.3592,  1.5922]],

        [[-0.2972,  0.0445, -0.4258,  ..., -1.3922,  0.2928, -0.1978],
         [-1.2154,  0.2450, -1.3642,  ..., -0.4660,  0.1048, -0.5017],
         [-0.4969,  1.1301,  1.2478,  ...,  1.8617, -1.0346,  0.5189],
         [ 1.6518,  0.3066, -0.8928,  ...,  0.0419,  0.5736,  1.3731],
         [ 1.2544,  0.2098, -0.6348,  ..., -0.2434,  0.0731,  0.3342]]],
       grad_fn=<IndexBackward0>)